Loading the models

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Normalization
from tensorflow.keras.initializers import Orthogonal
import numpy as np
import pandas as pd
import tensorflow as tf
from MockBroker import MockBroker
# import matplotlib.pyplot as plt
import plotly.graph_objects as go



models = {}
norm_layers = {}

nifty_symbols = [
    "ADANIPORTS", "ASIANPAINT", "AXISBANK", "BAJAJFINSV",
    "BAJFINANCE", "BPCL", "BRITANNIA", "CIPLA", "COALINDIA",
    "DIVISLAB", 
    "DRREDDY", "EICHERMOT", "GRASIM", "HCLTECH",
    "HDFCBANK", "HDFCLIFE", "HEROMOTOCO", "HINDALCO", 
    "HINDUNILVR",
    "ICICIBANK", "ICICIGI", "IOC", "INDUSINDBK", "INFY",
    "ITC", "JSWSTEEL", "KOTAKBANK", "LTTS", "LT",
    "MARICO",
      "MARUTI", "NESTLEIND"
]

# Load models
for symbol in nifty_symbols:
    models[symbol] = load_model(
        f'./models/{symbol}_model.keras', 
        # custom_objects={'Orthogonal': Orthogonal}
    )

data_test = {}
data_pred = {}
price = {}

threshold = 0.001
days= 500
cash =10000

for symbol in nifty_symbols:
    # Load the test data
    price[symbol] = pd.read_csv(f'./data/{symbol}.csv').iloc[-days:, 0].values
    data_test[symbol] = pd.read_csv(f'./data/{symbol}.csv').iloc[-days:, [0,1,2,3]].values

    # Normalize the test data
    norm_l = Normalization(axis=-1)
    norm_l.adapt(data_test[symbol])  # Ensure normalization layer adapts to the test data
    data_test_n = norm_l(data_test[symbol])

    # Reshape the data for LSTM input
    data_test_n = np.reshape(data_test_n, (data_test_n.shape[0], 1, data_test_n.shape[1]))

    # Predict using the model
    data_pred[symbol] = models[symbol].predict(data_test_n)

    






16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━

Buy only 1st

In [1]:


# Dictionary to store test data and predictions


# Load the test data
# Normalization layer (reusing the one from training)
# Initialize the broker
broker = MockBroker(cash, price)



for i in range(days):
    print(f"--- Day {i} ---")
    predictions = []

    for symbol in nifty_symbols:
        predictions.append((symbol, data_pred[symbol][i][0]))

    # Sort the predictions by the prediction value
    predictions.sort(key=lambda x: x[1], reverse=True)

    # Get the top 3 symbols with the highest prediction values
    top_3_symbols = predictions[:3]
    print(top_3_symbols)
    broker.sell_off(i)


    for symbol, prediction in top_3_symbols[:1]:
        current_price = price[symbol][i]
        qty = np.floor(broker.Balance / (current_price))
        if prediction > threshold:
            broker.buy(symbol, current_price, qty)

    print(broker.holdings)


x = np.linspace(0, days, days)  # 100 points between 0 and 10
y = broker.networth

fig = go.Figure()

# Add a line plot to the figure
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='lines',
    name='Net Worth',  # Legend label
    line=dict(color='blue', width=2)  # Line styling
))

# Add title and labels to the plot
fig.update_layout(
    title='Net Worth Over Time',
    xaxis_title='Days',
    yaxis_title='Net Worth',
    template='plotly_dark'  # Optional: Add a dark theme for the plot
)

# Show the plot
fig.show()

print(broker)



NameError: name 'MockBroker' is not defined

Divide by 3

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Normalization
from tensorflow.keras.initializers import Orthogonal
from MockBroker import MockBroker


# Initialize the broker



# Dictionary to store test data and predictions


broker = MockBroker(cash, price)


# Load the test data
# Normalization layer (reusing the one from training)




for i in range(days):
    print(f"--- Day {i} ---")
    predictions = []

    for symbol in nifty_symbols:
        predictions.append((symbol, data_pred[symbol][i][0]))

    # Sort the predictions by the prediction value
    predictions.sort(key=lambda x: x[1], reverse=True)

    # Get the top 3 symbols with the highest prediction values
    top_3_symbols = predictions[:3]
    print(top_3_symbols)
    broker.sell_off(i)



    for symbol, prediction in top_3_symbols[:3]:
        current_price = price[symbol][i]
        qty = np.floor(broker.Balance / (3*current_price))
        if prediction > threshold:
            broker.buy(symbol, current_price, qty)
    
    print(broker.holdings)



x = np.linspace(0, days, days)  # 100 points between 0 and 10
y = broker.networth

fig = go.Figure()

# Add a line plot to the figure
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='lines',
    name='Net Worth',  # Legend label
    line=dict(color='blue', width=2)  # Line styling
))

# Add title and labels to the plot
fig.update_layout(
    title='Net Worth Over Time',
    xaxis_title='Days',
    yaxis_title='Net Worth (USD)',
    template='plotly_dark'  # Optional: Add a dark theme for the plot
)

# Show the plot
fig.show()

print(broker)


--- Day 0 ---
[('COALINDIA', 0.0034776789), ('LT', 0.0027207125), ('JSWSTEEL', 0.0025995576)]
Networth: 10000
Bought 15.0 of COALINDIA at 218.14999 each
Balance: 6727.75015
Bought 1.0 of LT at 2087.3501 each
Balance: 4640.40005
Bought 2.0 of JSWSTEEL at 750.70001 each
Balance: 3139.00003
{'COALINDIA': 15, 'LT': 1, 'JSWSTEEL': 2}
--- Day 1 ---
[('COALINDIA', 0.0036299757), ('JSWSTEEL', 0.0028822015), ('LT', 0.0027510384)]
Sold 15 of COALINDIA at 214.25 each
Balance: 6352.75003
Sold 1 of LT at 2086.55 each
Balance: 8439.30003
Sold 2 of JSWSTEEL at 728.34998 each
Balance: 9895.99999
Networth: 9895.99999
Bought 15.0 of COALINDIA at 214.25 each
Balance: 6682.24999
Bought 3.0 of JSWSTEEL at 728.34998 each
Balance: 4497.20005
Bought 0.0 of LT at 2086.55 each
Balance: 4497.20005
{'COALINDIA': 15, 'LT': 0, 'JSWSTEEL': 3}
--- Day 2 ---
[('COALINDIA', 0.003854965), ('LT', 0.0028958367), ('ADANIPORTS', 0.0018840786)]
Sold 15 of COALINDIA at 214.95 each
Balance: 7721.45005
Sold 3 of JSWSTEEL at 742

Equity: 0.0
 Holdings: {'COALINDIA': 12, 'LT': 0, 'JSWSTEEL': 0, 'ADANIPORTS': 2, 'NESTLEIND': 0, 'MARICO': 0, 'ITC': 0, 'EICHERMOT': 0, 'ICICIGI': 0, 'HDFCLIFE': 0, 'BAJAJFINSV': 0, 'DIVISLAB': 0, 'HCLTECH': 0, 'CIPLA': 0, 'DRREDDY': 0, 'IOC': 0, 'KOTAKBANK': 0, 'INDUSINDBK': 0, 'BRITANNIA': 0, 'ASIANPAINT': 0}
 Balance: 7344.370940550023
 Total: 13990.770940550023
 Highest Net Worth: 16097.73107368001
 Profit off Stocks: {'COALINDIA': -2112.0, 'LT': 909.0, 'JSWSTEEL': 261.0, 'ADANIPORTS': -2195.4, 'NESTLEIND': 1.0, 'MARICO': 279.0, 'ITC': 7.0, 'EICHERMOT': -225.0, 'ICICIGI': 256.0, 'HDFCLIFE': 27.0, 'BAJAJFINSV': -119.0, 'DIVISLAB': 0.0, 'HCLTECH': 334.0, 'CIPLA': 99.0, 'DRREDDY': 31.0, 'IOC': 16.0, 'KOTAKBANK': 11.0, 'INDUSINDBK': -6.0, 'BRITANNIA': 0.0, 'ASIANPAINT': 10.0}


Now Shorting Divide by 6 stat

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Normalization
from tensorflow.keras.initializers import Orthogonal
from MockBroker import MockBroker


# Initialize the broker



# Dictionary to store test data and predictions


broker = MockBroker(cash, price)


# Load the test data
# Normalization layer (reusing the one from training)




for i in range(days):
    print(f"--- Day {i} ---")
    predictions = []

    for symbol in nifty_symbols:
        predictions.append((symbol, data_pred[symbol][i][0]))

    # Sort the predictions by the prediction value
    predictions.sort(key=lambda x: x[1], reverse=True)

    # Get the top 3 symbols with the highest prediction values
    top_3_symbols = predictions[:3]
    print(top_3_symbols)
    least_3_symbols = predictions[-3:]
    print(least_3_symbols)
    broker.sell_off(i)



    for symbol, prediction in top_3_symbols:
        current_price = price[symbol][i]
        qty = np.floor(broker.Balance / (6*current_price))
        if prediction > threshold:
            broker.buy(symbol, current_price, qty)
    
    for symbol, prediction in least_3_symbols:
        current_price = price[symbol][i]
        qty = np.floor(broker.Balance / (6*current_price))
        if prediction > threshold:
            broker.shortBuy(symbol, current_price, qty)
    print(broker.holdings)



x = np.linspace(0, days, days)  # 100 points between 0 and 10
y = broker.networth

fig = go.Figure()

# Add a line plot to the figure
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='lines',
    name='Net Worth',  # Legend label
    line=dict(color='blue', width=2)  # Line styling
))

# Add title and labels to the plot
fig.update_layout(
    title='Net Worth Over Time',
    xaxis_title='Days',
    yaxis_title='Net Worth (USD)',
    template='plotly_dark'  # Optional: Add a dark theme for the plot
)

# Show the plot
fig.show()

print(broker)


--- Day 0 ---
[('COALINDIA', 0.0034776789), ('LT', 0.0027207125), ('JSWSTEEL', 0.0025995576)]
[('ICICIBANK', -0.0025425095), ('ICICIGI', -0.0027456372), ('INDUSINDBK', -0.0043769507)]
Networth: 10000
Bought 7.0 of COALINDIA at 218.14999 each
Balance: 8472.950069999999
Bought 0.0 of LT at 2087.3501 each
Balance: 8472.950069999999
Bought 1.0 of JSWSTEEL at 750.70001 each
Balance: 7722.2500599999985
{'COALINDIA': 7, 'LT': 0, 'JSWSTEEL': 1}
--- Day 1 ---
[('COALINDIA', 0.0036299757), ('JSWSTEEL', 0.0028822015), ('LT', 0.0027510384)]
[('ICICIBANK', -0.0025226693), ('ICICIGI', -0.0027222792), ('INDUSINDBK', -0.0034510659)]
Sold 7 of COALINDIA at 214.25 each
Balance: 9222.000059999998
Sold 1 of JSWSTEEL at 728.34998 each
Balance: 9950.350039999998
Networth: 9950.350039999998
Bought 7.0 of COALINDIA at 214.25 each
Balance: 8450.600039999998
Bought 1.0 of JSWSTEEL at 728.34998 each
Balance: 7722.250059999998
Bought 0.0 of LT at 2086.55 each
Balance: 7722.250059999998
{'COALINDIA': 7, 'LT': 0, '

Equity: 0.0
 Holdings: {'COALINDIA': 5, 'LT': 0, 'JSWSTEEL': 0, 'ADANIPORTS': 1, 'NESTLEIND': 0, 'MARICO': 0, 'ITC': 0, 'EICHERMOT': 0, 'ICICIGI': 0, 'HDFCLIFE': 0, 'BAJAJFINSV': 0, 'DIVISLAB': 0, 'HCLTECH': 0, 'CIPLA': 0, 'DRREDDY': 0, 'IOC': 0, 'KOTAKBANK': 0, 'INDUSINDBK': 0, 'BRITANNIA': 0, 'ASIANPAINT': 0}
 Balance: 8800.564527400025
 Total: 11755.264527400026
 Highest Net Worth: 12431.768818150027
 Profit off Stocks: {'COALINDIA': -785.5, 'LT': 0.0, 'JSWSTEEL': 125.0, 'ADANIPORTS': -1128.2, 'NESTLEIND': 0.0, 'MARICO': 142.0, 'ITC': 22.0, 'EICHERMOT': 0.0, 'ICICIGI': 268.0, 'HDFCLIFE': 20.0, 'BAJAJFINSV': 120.0, 'DIVISLAB': 0.0, 'HCLTECH': 114.0, 'CIPLA': -5.0, 'DRREDDY': 19.0, 'IOC': 39.0, 'KOTAKBANK': 7.0, 'INDUSINDBK': -2.0, 'BRITANNIA': 0.0, 'ASIANPAINT': 0.0}


Divide Balance in ratio

In [ ]:

# Dictionary to store test data and predictions


broker = MockBroker(cash, price)


# Load the test data
# Normalization layer (reusing the one from training)


for i in range(days):
    print(f"--- Day {i} ---")
    predictions = []

    for symbol in nifty_symbols:
        predictions.append((symbol, data_pred[symbol][i][0]))

    # Sort the predictions by the prediction value
    predictions.sort(key=lambda x: x[1], reverse=True)

    # Get the top 3 symbols with the highest prediction values
    top_3_symbols = predictions[:2]
    print(top_3_symbols)
    broker.sell_off(i)

    
    sumprices = 0

    for symbol, prediction in top_3_symbols[:3]:
            sumprices += price[symbol][i]
    
    # print(sumprices)

    for symbol, prediction in top_3_symbols[:3]:
        current_price = price[symbol][i]
        balance_alloted = broker.Balance*current_price/sumprices
        qty = np.floor(balance_alloted / (current_price))
        if prediction > threshold:
            broker.buy(symbol, current_price, qty)

    print(broker.holdings)


x = np.linspace(0, days, days)  # 100 points between 0 and 10
y = broker.networth

fig = go.Figure()

# Add a line plot to the figure
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='lines',
    name='Net Worth',  # Legend label
    line=dict(color='blue', width=2)  # Line styling
))

# Add title and labels to the plot
fig.update_layout(
    title='Net Worth Over Time',
    xaxis_title='Days',
    yaxis_title='Net Worth (USD)',
    template='plotly_dark'  # Optional: Add a dark theme for the plot
)

# Show the plot
fig.show()

print(broker)


--- Day 0 ---
[('COALINDIA', 0.0034776789), ('LT', 0.0027207125)]
Networth: 10000
Bought 4.0 of COALINDIA at 218.14999 each
Balance: 9127.40004
Bought 3.0 of LT at 2087.3501 each
Balance: 2865.3497399999997
{'COALINDIA': 4, 'LT': 3}
--- Day 1 ---
[('COALINDIA', 0.0036299757), ('JSWSTEEL', 0.0028822015)]
Sold 4 of COALINDIA at 214.25 each
Balance: 3722.3497399999997
Sold 3 of LT at 2086.55 each
Balance: 9981.99974
Networth: 9981.99974
Bought 10.0 of COALINDIA at 214.25 each
Balance: 7839.499739999999
Bought 8.0 of JSWSTEEL at 728.34998 each
Balance: 2012.6998999999996
{'COALINDIA': 10, 'LT': 0, 'JSWSTEEL': 8}
--- Day 2 ---
[('COALINDIA', 0.003854965), ('LT', 0.0028958367)]
Sold 10 of COALINDIA at 214.95 each
Balance: 4162.1999
Sold 8 of JSWSTEEL at 742.34998 each
Balance: 10100.99974
Networth: 10100.99974
Bought 4.0 of COALINDIA at 214.95 each
Balance: 9241.19974
Bought 3.0 of LT at 2121.7 each
Balance: 2876.0997400000006
{'COALINDIA': 4, 'LT': 3, 'JSWSTEEL': 0}
--- Day 3 ---
[('COALIND

Equity: 0.0
 Holdings: {'COALINDIA': 2, 'LT': 0, 'JSWSTEEL': 0, 'ICICIGI': 0, 'NESTLEIND': 0, 'EICHERMOT': 0, 'HDFCLIFE': 0, 'HCLTECH': 0, 'DRREDDY': 0, 'CIPLA': 0, 'BAJAJFINSV': 0, 'ITC': 0, 'ADANIPORTS': 0, 'MARICO': 0, 'IOC': 0, 'KOTAKBANK': 0, 'BRITANNIA': 0, 'ASIANPAINT': 0, 'DIVISLAB': 2}
 Balance: 4529.763236729985
 Total: 16809.463236729986
 Highest Net Worth: 19556.06274924
 Profit off Stocks: {'COALINDIA': 442.0, 'LT': 3247.0, 'JSWSTEEL': 939.0, 'ICICIGI': 727.0, 'NESTLEIND': -42.0, 'EICHERMOT': 1581.0, 'HDFCLIFE': 3.0, 'HCLTECH': 1214.0, 'DRREDDY': -168.0, 'CIPLA': -1067.0, 'BAJAJFINSV': -919.0, 'ITC': 215.0, 'ADANIPORTS': 130.0, 'MARICO': 103.0, 'IOC': 4.0, 'KOTAKBANK': -179.0, 'BRITANNIA': 57.0, 'ASIANPAINT': 63.0, 'DIVISLAB': -11729.7}
